# 02 — Train XGBoost Models (Multi-Outcome)

Trains one XGBoost classifier per instability outcome using the feature sets
selected by Notebook `03/01`. One model per outcome — no stacking.

## Relationship to existing classifier notebook

| Component | Treatment |
|---|---|
| `temporal_split()` | Reused (train ≤2018, val 2019–2021, test ≥2022) |
| `RandomizedSearchCV` loop | Reused, one call per outcome with temporal CV |
| SHAP TreeExplainer | Reused, one call per outcome |
| MLflow logging | Reused, one `mlflow.start_run()` per outcome |
| Data loading | Reads ADLS parquet (not raw CSV) |
| Feature engineering | Done in Notebooks `02/02`–`02/03` — not repeated here |
| StratifiedKFold | **Replaced** with temporal CV to prevent future-leakage |

## Class imbalance

| Outcome | Expected base rate | `scale_pos_weight` |
|---|---|---|
| `civil_war_onset` | ~0.2% | ~500 |
| `coup_attempt` | ~0.1–0.3% | ~300–1000 |
| `regime_backsliding` | ~1% | ~100 |
| `mass_unrest_onset` | ~12–20% | ~4–8 |
| `humanitarian_crisis_onset` | ~5–10% (regional) | ~10–20 |

Primary metric: **AUPRC** (average precision). AUROC also logged.

## ADLS outputs per outcome
```
models/{RUN_DATE}/{outcome}/model.json
models/{RUN_DATE}/{outcome}/shap_values.parquet
```

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER       (default: 'data')
AZUREML_MLFLOW_URI   (optional)
```

In [ ]:
import os
import json
import re
import tempfile
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import RandomizedSearchCV, BaseCrossValidator
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.calibration import CalibrationDisplay

import xgboost as xgb
import shap

from azure.identity import DefaultAzureCredential
import adlfs
import mlflow
import mlflow.xgboost

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# Temporal split boundaries (must match notebooks 14 and 15)
TRAIN_END_YEAR    = 2018
VAL_END_YEAR      = 2021

OUTCOMES = [
    "civil_war_onset",
    "coup_attempt",
    "regime_backsliding",
    "mass_unrest_onset",
    "humanitarian_crisis_onset",
]

# Hyperparameter search
N_ITER_SEARCH     = 40   # reduced to 20 for humanitarian (small N)
N_ITER_HUMANITARIAN = 20
RANDOM_STATE      = 42

# XGBoost base params (overridden by search)
XGB_BASE_PARAMS = {
    "objective":     "binary:logistic",
    "eval_metric":   "aucpr",
    "tree_method":   "hist",
    "random_state":  RANDOM_STATE,
    "n_jobs":        -1,
    "verbosity":     0,
}

# Hyperparameter search space
PARAM_DIST = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [3, 4, 5, 6],
    "learning_rate":     [0.01, 0.05, 0.1, 0.2],
    "subsample":         [0.6, 0.7, 0.8, 1.0],
    "colsample_bytree":  [0.6, 0.7, 0.8, 1.0],
    "min_child_weight":  [1, 3, 5, 10],
    "gamma":             [0, 0.1, 0.3, 1.0],
    "reg_alpha":         [0, 0.01, 0.1, 1.0],
    "reg_lambda":        [1.0, 2.0, 5.0],
}

print(f"Run date       : {RUN_DATE}")
print(f"Temporal split : train ≤{TRAIN_END_YEAR} | val ≤{VAL_END_YEAR} | test ≥{VAL_END_YEAR+1}")

## ADLS helpers and data loading

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential":   credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

def read_latest_parquet(prefix: str) -> pd.DataFrame | None:
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    full_prefix = f"{ADLS_CONTAINER}/{prefix}"
    try:
        entries = fs.ls(full_prefix, detail=False)
    except FileNotFoundError:
        print(f"  WARNING: prefix not found: {full_prefix}")
        return None
    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)], reverse=True
    )
    if not date_dirs:
        return None
    latest_dir = date_dirs[0]
    files = [
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + f.replace(f"{ADLS_CONTAINER}/", "", 1)
        for f in fs.glob(f"{latest_dir}/*.parquet")
    ]
    if not files:
        return None
    dfs = [pd.read_parquet(p, storage_options=storage_options) for p in files]
    return pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]

def read_json_from_adls(subpath: str) -> dict | None:
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    path = f"{ADLS_CONTAINER}/{subpath}"
    try:
        with fs.open(path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"  WARNING: not found: {path}")
        return None

def write_xgb_model(model: xgb.XGBClassifier, subpath: str) -> None:
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as tmp:
        model.save_model(tmp.name)
        with open(tmp.name, "rb") as src:
            with fs.open(f"{ADLS_CONTAINER}/{subpath}", "wb") as dst:
                dst.write(src.read())
    print(f"  Model saved → {subpath}")

# ── Load feature matrix and labels ───────────────────────────────────────────
fs = adlfs.AzureBlobFileSystem(account_name=ADLS_ACCOUNT_NAME, credential=credential)
_prefix = f"{ADLS_CONTAINER}/processed/feature_matrix"
_entries = fs.ls(_prefix, detail=False)
_date_dirs = sorted([e for e in _entries if re.search(r'/\d{8}(/|$)', e)], reverse=True)

def _read_named(filename):
    files = fs.glob(f"{_date_dirs[0]}/{filename}")
    if not files:
        return None
    path = (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + files[0].replace(f"{ADLS_CONTAINER}/", "", 1)
    )
    return pd.read_parquet(path, storage_options=storage_options)

df_features = _read_named("feature_matrix.parquet")
df_labels   = _read_named("labels.parquet")

OUTCOME_COLS = [
    "civil_war_onset", "coup_attempt", "regime_backsliding",
    "mass_unrest_onset", "humanitarian_crisis_onset",
]
ID_COLS = ["iso3", "year"]

if df_features is not None and df_labels is not None:
    print(f"Feature matrix : {df_features.shape}")
    print(f"Labels         : {df_labels.shape}")
else:
    print("ERROR: could not load from ADLS")

## Temporal split and data preparation

In [ ]:
def temporal_split(df: pd.DataFrame, outcome: str):
    """
    Split a country-year panel into train / val / test by year.

    Returns:
        X_train, y_train, X_val, y_val, X_test, y_test
        (all numpy arrays; feature names preserved in feat_cols)
    """
    feat_cols = [c for c in df.columns if c not in ID_COLS + OUTCOME_COLS]

    train = df[df["year"] <= TRAIN_END_YEAR]
    val   = df[(df["year"] > TRAIN_END_YEAR) & (df["year"] <= VAL_END_YEAR)]
    test  = df[df["year"] > VAL_END_YEAR]

    def _xy(subset):
        X = subset[feat_cols].values.astype(float)
        y = subset[outcome].values.astype(int)
        return X, y

    X_tr, y_tr = _xy(train)
    X_v,  y_v  = _xy(val)
    X_te, y_te = _xy(test)
    return X_tr, y_tr, X_v, y_v, X_te, y_te, feat_cols


def load_selected_features(outcome: str) -> list[str] | None:
    """Load the feature manifest written by Notebook 15."""
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    prefix = f"{ADLS_CONTAINER}/feature_selection"
    try:
        entries = fs.ls(prefix, detail=False)
    except FileNotFoundError:
        return None
    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)], reverse=True
    )
    if not date_dirs:
        return None
    path = f"{date_dirs[0]}/selected_{outcome}.json"
    try:
        with fs.open(path, "r") as f:
            manifest = json.load(f)
        return manifest.get("feature_names", [])
    except FileNotFoundError:
        print(f"  WARNING: no feature manifest for {outcome}")
        return None


def prepare_outcome_df(outcome: str, fews_countries: set | None = None):
    """
    Merge features + labels, restrict to valid rows, apply selected features.

    Returns merged DataFrame ready for temporal_split(), or None.
    """
    features = load_selected_features(outcome)
    if features is None:
        print(f"  {outcome}: no feature manifest — using all columns")
        feat_cols = [c for c in df_features.columns if c not in ID_COLS + OUTCOME_COLS]
    else:
        # Keep only features that actually exist in the matrix
        feat_cols = [f for f in features if f in df_features.columns]
        missing_f = [f for f in features if f not in df_features.columns]
        if missing_f:
            print(f"  {outcome}: {len(missing_f)} selected features not in matrix")

    merged = df_features[ID_COLS + feat_cols].merge(
        df_labels[["iso3", "year", outcome]],
        on=["iso3", "year"], how="inner",
    )
    merged = merged[merged[outcome].notna()].copy()

    if outcome == "humanitarian_crisis_onset" and fews_countries:
        merged = merged[merged["iso3"].isin(fews_countries)]

    # Median impute (computed on training rows only)
    train_mask = merged["year"] <= TRAIN_END_YEAR
    medians = merged.loc[train_mask, feat_cols].median()
    merged[feat_cols] = merged[feat_cols].fillna(medians)

    n_pos  = int((merged[outcome] == 1).sum())
    n_tot  = len(merged)
    print(f"  {outcome}: {n_tot:,} rows | {n_pos} positives ({n_pos/n_tot:.3f}) "
          f"| {len(feat_cols)} features")
    return merged, feat_cols


# Load FEWS country list
_cw_path = Path("../data/country_crosswalk.csv")
if not _cw_path.exists():
    _cw_path = Path("data/country_crosswalk.csv")
_cw = pd.read_csv(_cw_path, dtype=str)
FEWS_COUNTRIES = set(_cw.loc[_cw["fews_monitored"] == "1", "iso3"])

print("temporal_split, load_selected_features, prepare_outcome_df defined")

## Temporal CV for hyperparameter search

In [ ]:
class ExpandingYearCV(BaseCrossValidator):
    """Expanding-window temporal CV (same as Notebook 15)."""

    def __init__(self, years: np.ndarray, n_splits: int = 5, val_years: int = 2,
                 min_val_positives: int = 2):
        self.years             = years
        self.n_splits          = n_splits
        self.val_years         = val_years
        self.min_val_positives = min_val_positives

    def split(self, X, y=None, groups=None):
        unique_years = np.sort(np.unique(self.years))
        candidates   = unique_years[: -self.val_years]
        step         = max(1, len(candidates) // self.n_splits)
        cutpoints    = candidates[step - 1 :: step][: self.n_splits]
        for cutpoint in cutpoints:
            train_idx = np.where(self.years <= cutpoint)[0]
            val_idx   = np.where(
                (self.years > cutpoint) &
                (self.years <= cutpoint + self.val_years)
            )[0]
            if y is not None and len(val_idx):
                n_pos = np.sum(np.array(y)[val_idx] == 1)
                if n_pos < self.min_val_positives:
                    continue
            if len(train_idx) and len(val_idx):
                yield train_idx, val_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

print("ExpandingYearCV defined")

## Training loop — one XGBoost model per outcome

In [ ]:
mlflow_uri = os.getenv("AZUREML_MLFLOW_URI", "")
if mlflow_uri:
    mlflow.set_tracking_uri(mlflow_uri)
    mlflow.set_experiment("instability_xgboost")

model_results = {}

for outcome in OUTCOMES:
    print(f"\n{'='*60}")
    print(f"Outcome: {outcome}")
    print('='*60)

    fews = FEWS_COUNTRIES if outcome == "humanitarian_crisis_onset" else None

    try:
        df_out, feat_cols = prepare_outcome_df(outcome, fews)
    except Exception as e:
        print(f"  SKIPPED — {e}")
        continue

    X_tr, y_tr, X_v, y_v, X_te, y_te, _ = temporal_split(df_out, outcome)

    if y_tr.sum() < 5:
        print(f"  SKIPPED — too few training positives ({int(y_tr.sum())})")
        continue

    # scale_pos_weight: ratio of negatives to positives in training set
    spw = float((y_tr == 0).sum()) / max(int((y_tr == 1).sum()), 1)
    print(f"  scale_pos_weight = {spw:.1f}")

    # Temporal CV on training data
    train_years = df_out.loc[df_out["year"] <= TRAIN_END_YEAR, "year"].values
    cv = ExpandingYearCV(years=train_years, n_splits=5, val_years=2, min_val_positives=2)

    base_model = xgb.XGBClassifier(
        **XGB_BASE_PARAMS,
        scale_pos_weight=spw,
    )

    n_iter = N_ITER_HUMANITARIAN if outcome == "humanitarian_crisis_onset" else N_ITER_SEARCH

    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=PARAM_DIST,
        n_iter=n_iter,
        scoring="average_precision",
        cv=cv,
        refit=True,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0,
    )
    search.fit(X_tr, y_tr)

    best_params = search.best_params_
    best_model  = search.best_estimator_
    print(f"  Best CV AUPRC  = {search.best_score_:.4f}")
    print(f"  Best params    : {best_params}")

    # Evaluate on val and test
    val_auprc  = average_precision_score(y_v,  best_model.predict_proba(X_v)[:, 1])  if y_v.sum()  > 0 else float("nan")
    test_auprc = average_precision_score(y_te, best_model.predict_proba(X_te)[:, 1]) if y_te.sum() > 0 else float("nan")
    val_auroc  = roc_auc_score(y_v,  best_model.predict_proba(X_v)[:, 1])  if y_v.sum()  > 1 else float("nan")
    test_auroc = roc_auc_score(y_te, best_model.predict_proba(X_te)[:, 1]) if y_te.sum() > 1 else float("nan")
    print(f"  Val  AUPRC={val_auprc:.4f}  AUROC={val_auroc:.4f}")
    print(f"  Test AUPRC={test_auprc:.4f}  AUROC={test_auroc:.4f}")

    # SHAP TreeExplainer (test set, capped at 500 rows for memory)
    explainer   = shap.TreeExplainer(best_model)
    X_te_sample = X_te[:500]
    shap_values = explainer.shap_values(X_te_sample)

    shap_df = pd.DataFrame(shap_values, columns=feat_cols)
    shap_df["iso3"] = df_out.loc[df_out["year"] > VAL_END_YEAR, "iso3"].values[:500]
    shap_df["year"] = df_out.loc[df_out["year"] > VAL_END_YEAR, "year"].values[:500]
    shap_df["outcome"] = outcome

    # SHAP beeswarm
    fig_bee, ax_bee = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X_te_sample, feature_names=feat_cols,
                      show=False, max_display=20, plot_size=None)
    plt.title(f"SHAP beeswarm — {outcome}")
    plt.tight_layout()

    # Calibration plot
    fig_cal, ax_cal = plt.subplots(figsize=(6, 5))
    if y_te.sum() > 0:
        CalibrationDisplay.from_predictions(
            y_te, best_model.predict_proba(X_te)[:, 1],
            n_bins=10, ax=ax_cal, name=outcome,
        )
        ax_cal.set_title(f"Calibration — {outcome}")
        fig_cal.tight_layout()

    # Save model and SHAP values
    write_xgb_model(best_model, f"models/{RUN_DATE}/{outcome}/model.json")
    write_parquet(shap_df,      f"models/{RUN_DATE}/{outcome}/shap_values.parquet")

    # MLflow logging
    if mlflow_uri:
        with mlflow.start_run(run_name=outcome, tags={"outcome": outcome}):
            mlflow.log_params({**best_params, "scale_pos_weight": spw,
                               "n_features": len(feat_cols)})
            mlflow.log_metrics({
                "val_auprc":  val_auprc,  "val_auroc":  val_auroc,
                "test_auprc": test_auprc, "test_auroc": test_auroc,
            })
            mlflow.log_figure(fig_bee, f"shap_beeswarm_{outcome}.png")
            if y_te.sum() > 0:
                mlflow.log_figure(fig_cal, f"calibration_{outcome}.png")
            mlflow.xgboost.log_model(best_model, artifact_path=f"xgb_{outcome}")

    plt.show()
    plt.close("all")

    model_results[outcome] = {
        "val_auprc": val_auprc, "test_auprc": test_auprc,
        "val_auroc": val_auroc, "test_auroc": test_auroc,
        "n_features": len(feat_cols),
        "scale_pos_weight": spw,
    }

print(f"\n{'='*60}")
print("Training complete")
print(f"{'Outcome':<35} {'Val AUPRC':>10} {'Test AUPRC':>11}")
print("-" * 60)
for outcome, m in model_results.items():
    print(f"  {outcome:<33} {m['val_auprc']:>10.4f} {m['test_auprc']:>11.4f}")